In [1]:
# Importing Libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import scipy as sp
import numpy as np
import pandas as pd
import time
import datetime
import os
import sys
import pyvisa
import threading
import ipywidgets
import nidaqmx
import warnings
from scipy import signal
from collections import deque
from IPython.display import clear_output, display

print(f"All Libraries imported successfully at {datetime.datetime.now()}")

All Libraries imported successfully at 2026-04-16 14:31:43.109335


In [23]:
# Open Pyvisa Resource Manager and connect to insturements at 14T cryostat
rm = pyvisa.ResourceManager()

lakeshore = rm.open_resource("TCPIP0::192.168.150.121::7777::SOCKET", timeout=5000, write_termination='\n', read_termination='\r\n')
print(lakeshore.query("*IDN?"))

print(f"\nInstruments connected successfully at {datetime.datetime.now()}")

LSCI,MODEL372,LSA24KE,1.4

Instruments connected successfully at 2026-04-16 15:37:02.059146


In [ ]:
# Open Pyvisa Resource Manager and connect to insturements at Fincryo cryostat
rm = pyvisa.ResourceManager()

lakeshore = rm.open_resource("GPIB1::5::INSTR", timeout=5000)
print(lakeshore.query("*IDN?"))

print(f"\nInstruments connected successfully at {datetime.datetime.now()}")

In [ ]:
# Function to read temperature from Lakeshore 370/372 specific channel
def readtemp_lakeshore(address, channel):
    try:
        temp = lakeshore.query_ascii_values(f"RDGK? {channel}")[0]
    except Exception:
        temp = np.nan
    return temp

# Function to read Temperature Control Mode from Lakeshore 370/372
def readmode_lakeshore(address, model):
    try:
        if model == "370":
            n = lakeshore.query_ascii_values("CMODE?")[0]
            mode_list = ["Closed Loop PID", "Zone Tuning", "Open Loop", "Off"]
            mode = mode_list[int(n)-1]
        elif model == "372":
            n = lakeshore.query("OUTMODE? 0")[0]
            mode_list = ["Off", "Monitor Out", "Open Loop", "Zone", "Still", "Closed Loop PID", "Warm Up"]
            mode = mode_list[int(n)]
    except Exception:
        mode = "Error"
    return mode

# Function to read Heater Range from Lakeshore 370/372
def readrange_lakeshore(address, model):
    range_list = ["off","31.6 uA","100 uA","316 uA","1.00 mA","3.16 mA","10.0 mA","31.6 mA","100 mA"]
    try:
        if model == "370":
            n = lakeshore.query_ascii_values("HTRRNG?")[0]
            range = range_list[int(n)]
        elif model == "372":
            n = lakeshore.query_ascii_values("RANGE? 0")[0]
            range = range_list[int(n)]
    except Exception:
        range = "Error"
    return range

# Function to read Heater PID Parameters from Lakeshore 370/372
def readpid_lakeshore(address, model):
    try:
        if model == "370":
            params = lakeshore.query_ascii_values("PID?")
        elif model == "372":
            params = lakeshore.query_ascii_values("PID? 0")
    except Exception:
        params = [np.nan, np.nan, np.nan]
    return params

# Function to read Heater Status from Lakeshore 370/372
def readheater_lakeshore(address, model):
    try:
        mode = readmode_lakeshore(address, model)
        range = readrange_lakeshore(address, model)
        P, I, D = readpid_lakeshore(address, model)
        status = {"Mode": mode, "Range": range, "P": P, "I": I, "D": D}
    except Exception:
        status = np.nan
    return status

# Function to set Temperature Control Mode on Lakeshore 370/372
def setmode_lakeshore(address, model, mode):
    try:
        if model == "370":
            mode_list = ["Closed Loop PID", "Zone Tuning", "Open Loop", "Off"]
            if mode not in mode_list:
                print("Invalid Mode. Valid options are:", mode_list)
                return
            n = mode_list.index(mode) + 1
            lakeshore.write(f"CMODE {n}")
        elif model == "372":
            mode_list = ["Off", "Monitor Out", "Open Loop", "Zone", "Still", "Closed Loop PID", "Warm Up"]
            if mode not in mode_list:
                print("Invalid Mode. Valid options are:", mode_list)
                return
            n = mode_list.index(mode)
            lakeshore.write(f"OUTMODE 0,{n},6,1,0,1,60")
    except Exception:
        pass

# Function to set Heater Range on Lakeshore 370/372
def setrange_lakeshore(address, model, range):
    try:
        range_list = ["off","31.6 uA","100 uA","316 uA","1.00 mA","3.16 mA","10.0 mA","31.6 mA","100 mA"]
        if range not in range_list:
            print("Invalid Range. Valid options are:", range_list)
            return
        n = range_list.index(range)
        if model == "370":
            lakeshore.write(f"HTRRNG {n}")
        elif model == "372":
            lakeshore.write(f"RANGE 0,{n}")
    except Exception:
        pass

# Function to set Heater PID Parameters on Lakeshore 370/372
def setpid_lakeshore(address, model, P, I, D):
    try:
        if model == "370":
            lakeshore.write(f"PID {P},{I},{D}")
        elif model == "372":
            lakeshore.write(f"PID 0,{P},{I},{D}")
    except Exception:
        pass

# Function to set Temperature Setpoint on Lakeshore 370/372
def settemp_lakeshore(address, model, temp):
    try:
        if model == "370":
            lakeshore.write(f"SETP {temp}")
        elif model == "372":
            lakeshore.write(f"SETP 0,{temp}")
    except Exception:
        pass

def stabilizetemp_lakeshore(address, model, temp, tolerance=0.01, timeout=3600):
    try:
        settemp_lakeshore(address, model, temp)
        start_time = time.time()
        while time.time() - start_time < timeout:
            current_temp = readtemp_lakeshore(address, 6)
            print(f"Stabilizing {temp} K. Now {current_temp} K\r", end="")
            if abs(current_temp - temp) <= tolerance*temp:
                print(f"Temperature stabilized at {current_temp} K.")
                return True
            time.sleep(1)
        print(f"Failed to stabilize temperature within {timeout} seconds.")
    except Exception:
        pass

In [ ]:
setmode_lakeshore(lakeshore, "372", "Closed Loop PID")
setrange_lakeshore(lakeshore, "372", "3.16 mA")
print(readheater_lakeshore(lakeshore, "372"))
stabilizetemp_lakeshore(lakeshore, "372", 0.125, tolerance=0.01, timeout=3600)


{'Mode': 'Closed Loop PID', 'Range': '3.16 mA', 'P': 10.0, 'I': 20.0, 'D': 2.0}


KeyboardInterrupt: 